# GRPO on the SFT student

Round 3 of reinforcement learning: round 2's recipe, from the same start (SFT round 3, `qwen3.5-0.8b-sft-r3`), with the adapter kept every 20 steps and the best of them picked on a suite of its own.

1. **The train suite.** `train-rl`: 700 cases for 70 users, half the merchants and items invented (`make_suite.py --name train-rl --per-locale 100 --users-per-locale 10 --seed 2 --novel-share 0.5`, git-ignored, regenerated here and checked against its fingerprint). The trainer leaves out the 53 whose content a frozen suite repeats word for word, so 647 remain.
2. **GRPO.** `train_grpo.py` continues the round-3 SFT LoRA exactly as round 2 did, and keeps the adapter of steps 20, 40, 60 and 80 (`--keep-every 20`, as `step-020/` … `step-080/` beside the final adapter).
3. **Pick the checkpoint** on `val-rl`: 280 new cases for 70 new users (`--seed 3 --per-locale 40 --users-per-locale 10 --novel-share 0.5`), minus those whose content a frozen suite or `train-rl` repeats. Each kept step is merged and scored on it, and the one with the highest reward is round 3's model. SFT round 3 and GRPO round 2 are scored on it too, for reference: step 80 is round 2 rerun, so their gap is the run-to-run noise.
4. **Merge** the chosen step into a copy of the base checkpoint, for `baseline_gguf.ipynb` to quantize.
5. **Score it unquantized** on `dev` and `dev-novel` on vLLM, against SFT round 3, GRPO round 2 and the teacher.

**Why a validation suite, not `dev`.** Picking the best of four on `dev` and then reporting its `dev` score reports the luckiest of four draws. `val-rl` is drawn like the training data (half the shops invented, from the training half of the invented names) and is never trained on: the train scripts refuse any suite whose name doesn't start with `train`. So `dev` and `dev-novel` stay an honest measure of the one model that is picked.

| | GRPO round 3 |
|---|---|
| start | the SFT round-3 adapter (rank 16, on every linear layer) |
| step | 8 cases × 8 sessions at temperature 1.0, up to 32 requests generated together, one update |
| update | lr 1e-5; fields averaged per completion token, confidence per session (`CONFIDENCE_WEIGHT` 1); KL 0.04 to the start |
| steps | 80, the adapter kept every 20 |

The notebook clones `REPO_URL` at `BRANCH`, [kurexi/cost](https://github.com/kurexi/cost) by default; point them at a fork to run your own changes (for a private one, add a secret named `GITHUB_TOKEN` with read access to it). `AGENT` is the agent's folder inside the clone: empty for kurexi/cost, where the agent is the whole repository. `RECORDS` is the folder, inside the clone, that holds `runs/` and `RESULTS.md`; empty by default, which keeps them in the agent's `eval/` folder, where git ignores them. The eval scripts find it through `SPENDING_EVAL_RECORDS`.

The adapter is checkpointed every 5 steps, and the kept steps, the choice (`chosen.json`) and the merged model are kept beside it. If the session disconnects, run the cells again from the top: training resumes from its last checkpoint, finished steps are skipped, and a choice already made is kept.

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
from google.colab import userdata

def sh(args, cwd=None):
    process = subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args)


REPO_URL = "https://github.com/kurexi/cost"
BRANCH = "main"
RECORDS = ""
AGENT = ""
CLONE = "/content/repo"
EVAL = os.path.join(CLONE, AGENT, "eval")
RECORDS_DIR = f"{CLONE}/{RECORDS}" if RECORDS else EVAL
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = ""
url = REPO_URL.replace("https://", f"https://{token}@", 1) if token else REPO_URL
os.chdir("/content")
sh(["rm", "-rf", CLONE])
sh(["git", "clone", "--depth", "1", "--branch", BRANCH, url, CLONE])
os.environ["SPENDING_EVAL_RECORDS"] = RECORDS_DIR
os.makedirs(f"{RECORDS_DIR}/runs", exist_ok=True)

## Settings

`SFT_ROUND` is the adapter GRPO continues. `ROUND` names this run's adapter, merged model (`qwen3.5-0.8b-grpo-r3`) and scores; change it for another run so nothing is overwritten. `PREVIOUS` is the run its scores are compared with, and `EARLIER` the GRPO round before. `KEEP_EVERY` is how often the adapter is kept as a candidate, and `VAL_SUITE` the suite the candidates are judged on.

`KL` is the weight of the pull back to the starting adapter (0 drops it, as round 1 had). `CONFIDENCE_WEIGHT` is how much a session's confidence decision weighs against all its other tokens together.

In [ ]:
STUDENT = "Qwen/Qwen3.5-0.8B"
SFT_ROUND = "qwen3.5-0.8b-sft-r3"
ROUND = "r3"
NAME = f"qwen3.5-0.8b-grpo-{ROUND}"
PREVIOUS = SFT_ROUND
EARLIER = "qwen3.5-0.8b-grpo-r2"
TEACHER_NAME = "qwen3.8-27b-fp8-v5"
SUITE = "train-rl"
SUITE_FINGERPRINT = "3b7e550726f6949b"
STEPS = 80
CASES = 8
GROUP = 8
GEN_BATCH = 32
LR = 1e-5
TEMPERATURE = 1.0
KL = 0.04
CONFIDENCE_WEIGHT = 1.0
KEEP_EVERY = 20
VAL_SUITE = "val-rl"
VAL_FINGERPRINT = "99159aeb92055fbe"
print(NAME)

## Storage

The round-3 SFT adapter must already be there (`sft.ipynb` makes it). The GRPO adapter and its checkpoints go beside it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/spending-agent"
SFT_ADAPTER = f"{DRIVE}/sft/{SFT_ROUND}"
ADAPTER = f"{DRIVE}/grpo/{NAME}"
MERGED = f"{DRIVE}/models/{NAME}"
for folder in (ADAPTER, os.path.dirname(MERGED)):
    os.makedirs(folder, exist_ok=True)
if not os.path.exists(f"{SFT_ADAPTER}/adapter_model.safetensors"):
    raise SystemExit(f"no adapter at {SFT_ADAPTER}: run sft.ipynb for {SFT_ROUND} first")
print(sorted(os.listdir(ADAPTER)))


def finished(folder, cases):
    results = f"{folder}/results.jsonl"
    return os.path.exists(results) and sum(1 for _ in open(results)) >= cases

## The train suite

Generated from its seed; the fingerprint shows it is the same suite every time.

In [ ]:
import json
import shutil

os.chdir(EVAL)
subprocess.run(["python3", "make_suite.py", "--name", SUITE, "--per-locale", "100", "--users-per-locale", "10",
                "--seed", "2", "--novel-share", "0.5"], check=True, capture_output=True)
with open(f"suites/{SUITE}/meta.json") as f:
    suite = json.load(f)
print(suite["cases"], "cases, fingerprint", suite["fingerprint"])
if suite["fingerprint"] != SUITE_FINGERPRINT:
    raise SystemExit("the train suite doesn't regenerate as it did: the generator changed, so this is a new suite")

## Training packages and the base model

As for SFT: `transformers` new enough for Qwen3.5, `peft`, the `flash-linear-attention` kernels for the DeltaNet layers, `xgrammar`, which samples each answer within its JSON schema as vLLM does, and `jsonschema`, which grades an answer against the same bounds. The preinstalled `torchao` is removed, because `peft` refuses to attach a LoRA beside an older one. The base checkpoint is the one the SFT adapters were trained on.

`causal-conv1d` is the DeltaNet layers' short-convolution kernel. Without it `transformers` runs that convolution in plain PyTorch on every generated token, and generation is most of a GRPO step. pip takes a prebuilt wheel when one matches the installed torch and otherwise compiles it. The cell then imports it in a fresh process; a build that doesn't load is uninstalled again, so training runs on the slow path rather than failing.

In [ ]:
!pip install -q -U transformers peft flash-linear-attention jsonschema xgrammar
!pip uninstall -q -y torchao
sh(["pip", "install", "--no-build-isolation", "causal-conv1d"])
loads = subprocess.run(["python3", "-c", "from causal_conv1d import causal_conv1d_fn, causal_conv1d_update"],
                       capture_output=True, text=True)
if loads.returncode:
    print(loads.stderr[-2000:])
    sh(["pip", "uninstall", "-y", "causal-conv1d"])
print("causal-conv1d:", "fast kernel" if loads.returncode == 0 else "not loadable, PyTorch fallback")
from huggingface_hub import snapshot_download

BASE = "/content/models/qwen35-08b"
snapshot_download(STUDENT, local_dir=BASE)

## GRPO

Each step prints one line: the reward and `record_core` of its 64 sessions, false and right approvals, answers outside the schema (`outside_grammar`: with the grammar on, only answers cut off at the token limit, graded as no draft), tool calls, rounds, budget stops, the mean confidence on right and on wrong drafts, the KL to the starting adapter, how many sessions trained their confidence, and how many groups had any signal. The reward is noisy step to step: 8 cases is a small sample. Look at trends over tens of steps: `false_approval` and the gap between `confidence_right` and `confidence_wrong` are what this round is for, and `kl` should grow slowly rather than run away. A `kept step N` line follows every 20th step.

In [ ]:
import re

os.chdir(EVAL)
if not os.path.exists(f"{ADAPTER}/adapter_model.safetensors"):
    sh(["python3", "train_grpo.py", "--model", BASE, "--init-adapter", SFT_ADAPTER, "--suite", SUITE,
        "--out", ADAPTER, "--steps", str(STEPS), "--cases", str(CASES), "--group", str(GROUP),
        "--gen-batch", str(GEN_BATCH), "--lr", str(LR), "--temperature", str(TEMPERATURE), "--kl", str(KL),
        "--confidence-weight", str(CONFIDENCE_WEIGHT), "--save-every", "5", "--keep-every", str(KEEP_EVERY)])
!tail -n 3 {ADAPTER}/train_log.jsonl
KEPT = sorted(name for name in os.listdir(ADAPTER) if re.fullmatch(r"step-\d{3}", name))
print("kept:", KEPT)

## vLLM

In its own environment, the same server and settings as every bf16 row, so the differences are the models'.

In [ ]:
VLLM_VENV = "/content/vllm-venv"
if not os.path.exists(f"{VLLM_VENV}/bin/vllm"):
    sh(["pip", "install", "-q", "uv"])
    sh(["uv", "venv", "-q", "--python", "3.12", VLLM_VENV])
    sh(["uv", "pip", "install", "-q", "--python", f"{VLLM_VENV}/bin/python", "vllm"])
print(subprocess.run([f"{VLLM_VENV}/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                     capture_output=True, text=True).stdout)

In [ ]:
import time
import urllib.request


def stop_servers():
    subprocess.run("pkill -f '[v]llm serve'; pkill -f '[l]lama-server'", shell=True)
    for _ in range(60):
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout.strip()
        if int(used.split()[0]) < 1000:
            return
        time.sleep(5)
    raise SystemExit(f"the GPU still holds {used} MiB")


def log_tail(path, characters=4000):
    with open(path, errors="replace") as f:
        return f.read()[-characters:]


def serve(model, extra=""):
    """Start vLLM on `model` and wait for it, printing its latest log line
    every 30 seconds; stop at once, with the end of the log, if it exits."""
    stop_servers()
    log_path = "/content/vllm.log"
    process = subprocess.Popen([f"{VLLM_VENV}/bin/vllm", "serve", model, "--served-model-name", "qwen",
                                "--max-model-len", "16384", "--max-num-seqs", "32", "--gpu-memory-utilization", "0.90",
                                "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
                                "--reasoning-parser", "qwen3", "--port", "8090", *extra.split()],
                               stdout=open(log_path, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    started = reported = time.time()
    while True:
        if process.poll() is not None:
            raise SystemExit(f"vLLM exited with code {process.returncode}:\n{log_tail(log_path)}")
        try:
            urllib.request.urlopen("http://127.0.0.1:8090/v1/models", timeout=5)
            print(f"server is up after {time.time() - started:.0f} s")
            return
        except Exception:
            pass
        if time.time() - started > 2400:
            raise SystemExit(f"vLLM isn't up after 40 minutes:\n{log_tail(log_path)}")
        if time.time() - reported >= 30:
            lines = log_tail(log_path, 1000).strip().splitlines()
            print(f"{time.time() - started:.0f} s, still starting: {lines[-1] if lines else '(no log yet)'}", flush=True)
            reported = time.time()
        time.sleep(2)

## Pick the checkpoint on `val-rl`

Each kept step is merged into a scratch copy of the base checkpoint, served and run on `val-rl`; SFT round 3 and GRPO round 2 are run beside them for reference. A run already finished is kept, so a rerun only scores what is missing. The table gives each one's reward (with its 95% interval), `record_core`, false and right approvals, how many it approves and its approval precision; the step with the highest reward is chosen and written to `chosen.json` beside the adapter. A choice already made is kept, so the merged model can't change under a rerun. With 240 cases the intervals are about ±0.03 wide: steps whose rewards differ by less than that are a tie, and the choice between them is luck.

In [ ]:
os.chdir(EVAL)
subprocess.run(["python3", "make_suite.py", "--name", VAL_SUITE, "--per-locale", "40", "--users-per-locale", "10",
                "--seed", "3", "--novel-share", "0.5"], check=True, capture_output=True)
with open(f"suites/{VAL_SUITE}/meta.json") as f:
    if json.load(f)["fingerprint"] != VAL_FINGERPRINT:
        raise SystemExit("the validation suite doesn't regenerate as it did: the generator changed")


def contents(name):
    with open(f"suites/{name}/cases.jsonl", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


seen = {case["content"] for name in ("dev", "dev-novel", "holdout", SUITE) for case in contents(name)}
VAL_IDS = [case["id"] for case in contents(VAL_SUITE) if case["content"] not in seen]
print(len(VAL_IDS), "validation cases")

CANDIDATE = "/content/candidate"
RUN_ARGS = ["--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen", "--reasoning-effort", "none", "--workers", "16"]
adapters = {f"{NAME}-{step}-{VAL_SUITE}": f"{ADAPTER}/{step}" for step in KEPT}
step_of = {f"{NAME}-{step}-{VAL_SUITE}": step for step in KEPT}
references = {f"{SFT_ROUND}-{VAL_SUITE}": SFT_ADAPTER, f"{EARLIER}-{VAL_SUITE}": f"{DRIVE}/grpo/{EARLIER}"}
references = {run: adapter for run, adapter in references.items()
              if os.path.exists(f"{adapter}/adapter_model.safetensors")}
for run, adapter in {**references, **adapters}.items():
    if finished(f"{RECORDS_DIR}/runs/{run}", len(VAL_IDS)):
        continue
    print(f"== {run}: {adapter}", flush=True)
    shutil.rmtree(CANDIDATE, ignore_errors=True)
    sh(["python3", "merge_lora.py", "--base", BASE, "--adapter", adapter, "--out", CANDIDATE])
    serve(CANDIDATE, "--dtype bfloat16")
    sh(["python3", "run.py", "--suite", VAL_SUITE, "--name", run, *RUN_ARGS, "--ids", *VAL_IDS])
stop_servers()


def overall(run):
    with open(f"{RECORDS_DIR}/runs/{run}/grade.json") as f:
        return json.load(f)["summary"]["overall"]


print(f"{'run':42} {'reward':>21} {'core':>6} {'false':>6} {'right':>6} {'approves':>9} {'precision':>9}")
for run in [*references, *adapters]:
    o = overall(run)
    approves = o["false_approval"] + o["right_approval"]
    low, high = o["reward_ci95"]
    print(f"{run:42} {o['reward']:.3f} ({low:.3f}–{high:.3f}) {o['record_core']:6.3f} {o['false_approval']:6.3f} "
          f"{o['right_approval']:6.3f} {approves:9.0%} {o['right_approval'] / max(approves, 1e-9):9.2f}")

CHOSEN = f"{ADAPTER}/chosen.json"
if not os.path.exists(CHOSEN):
    best = max(adapters, key=lambda run: overall(run)["reward"])
    with open(CHOSEN, "w") as f:
        json.dump({"step": step_of[best], "val_suite": VAL_SUITE, "val_fingerprint": VAL_FINGERPRINT, "val_cases": len(VAL_IDS),
                   "rewards": {run: overall(run)["reward"] for run in [*references, *adapters]}}, f, indent=1)
with open(CHOSEN) as f:
    chosen = json.load(f)
print("chosen:", chosen["step"])

## Merge the chosen step

The chosen step's adapter is the round-3 SFT adapter trained further, so it is merged into the base checkpoint the same way. A merged model of another step (an earlier choice) is replaced.

In [ ]:
os.chdir(EVAL)
CHOSEN_ADAPTER = f"{ADAPTER}/{chosen['step']}"
merged_from = None
if os.path.exists(f"{MERGED}/merge.json"):
    with open(f"{MERGED}/merge.json") as f:
        merged_from = json.load(f)["adapter"]
if merged_from != CHOSEN_ADAPTER:
    shutil.rmtree(MERGED, ignore_errors=True)
    sh(["python3", "merge_lora.py", "--base", BASE, "--adapter", CHOSEN_ADAPTER, "--out", MERGED])
!ls -la {MERGED}

## Score it unquantized on `dev` and `dev-novel`

Each suite is compared with SFT round 3, GRPO round 2 and the teacher (first named = baseline). Each run also prints its false approvals, its mean confidence on right and on wrong drafts, its approval precision (the reward is built around 0.9), and its tool use.

In [ ]:
serve(MERGED, "--dtype bfloat16")
os.chdir(EVAL)
RUN_ARGS = ["--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen", "--reasoning-effort", "none", "--workers", "16"]
sh(["python3", "run.py", "--suite", "dev", "--name", f"{NAME}-bf16", *RUN_ARGS])
sh(["python3", "run.py", "--suite", "dev-novel", "--name", f"{NAME}-bf16-dev-novel", *RUN_ARGS])
stop_servers()
for lineup in ([f"{PREVIOUS}-bf16", f"{EARLIER}-bf16", f"{NAME}-bf16", TEACHER_NAME],
               [f"{PREVIOUS}-bf16-dev-novel", f"{EARLIER}-bf16-dev-novel", f"{NAME}-bf16-dev-novel",
                f"{TEACHER_NAME}-dev-novel"]):
    present = [run for run in lineup if finished(f"{RECORDS_DIR}/runs/{run}", 280)]
    if len(present) >= 2:
        sh(["python3", "compare.py", *present])
    for run in present:
        with open(f"{RECORDS_DIR}/runs/{run}/grade.json") as f:
            summary = json.load(f)["summary"]
        process = summary["process"]
        print(f"{run}: false approvals {summary['overall']['false_approval']:.3f}, "
              f"{process['rounds']:.2f} model rounds a case, budget spent in {process['budget_spent']:.0%}, "
              "tool calls a case by name", {name: round(n, 2) for name, n in process["tool_calls_by_name"].items()})
        with open(f"{RECORDS_DIR}/runs/{run}/grade.json") as f:
            cases = json.load(f)["per_case"]
        cases = list(cases.values()) if isinstance(cases, dict) else cases
        drafts = [c for c in cases if c["answered"] and isinstance(c.get("confidence"), (int, float))]
        right = [c["confidence"] for c in drafts if c["record_all"]]
        wrong = [c["confidence"] for c in drafts if not c["record_all"]]
        approved = [c for c in cases if c["would_approve"]]
        print(f"   confidence {sum(right) / max(1, len(right)):.3f} on right drafts, {sum(wrong) / max(1, len(wrong)):.3f} "
              f"on wrong ones; approval precision {sum(c['record_all'] for c in approved) / max(1, len(approved)):.2f} "
              f"over {len(approved)} approved")
sh(["python3", "board.py", "--write"])

## Take the results home

The zip holds what is worth keeping: the train log and arguments, the adapter's config, the choice, the merge report, the `val-rl` runs, the two bf16 runs and `RESULTS.md`. Unpack it into your records folder; none of it is committed, as training isn't part of the benchmark.

In [ ]:
OUT = "/content/out"
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(f"{OUT}/grpo/{NAME}")
for source in (f"{ADAPTER}/adapter_config.json", f"{ADAPTER}/train_args.json", f"{ADAPTER}/train_log.jsonl",
               CHOSEN, f"{MERGED}/merge.json"):
    shutil.copy(source, f"{OUT}/grpo/{NAME}/")
for run in (f"{NAME}-bf16", f"{NAME}-bf16-dev-novel", *references, *adapters):
    if finished(f"{RECORDS_DIR}/runs/{run}", 1):
        shutil.copytree(f"{RECORDS_DIR}/runs/{run}", f"{OUT}/runs/{run}")
shutil.copy(f"{RECORDS_DIR}/RESULTS.md", OUT)
subprocess.run(f"cd {OUT} && zip -qr /content/{NAME}.zip .", shell=True, check=True)
from google.colab import files
files.download(f"/content/{NAME}.zip")